# Skin Lesion Classification with Deep Learning

## Project overview
This notebook develops and evaluates image-classification models for the seven diagnostic categories in the HAM10000 dermoscopic image dataset. It includes dataset inspection, lesion-aware splitting, CNN baselines, DINOv2 feature extraction and fine-tuning, held-out evaluation, and explainability visualizations.

**Models:** ResNet-50, EfficientNet-B0, and DINOv2 (frozen-feature and fine-tuned configurations).

**Important:** This is an educational research prototype, not a clinically validated diagnostic system. Predictions must not be used to diagnose or rule out skin cancer.

### How to use this notebook
1. Run cells from top to bottom in Google Colab.
2. Mount Drive and update `DATA_DIR` if your dataset folder has a different name.
3. Enable a GPU for training when available.
4. Keep the test set for final evaluation only.
5. The training cells can take time; checkpoints are saved to Drive.

### Project workflow
1. Environment setup and dataset inspection
2. Cleaning, splitting, preprocessing, and data loaders
3. ResNet-50 baseline training and evaluation
4. Frozen DINOv2 feature extraction and classifier evaluation
5. DINOv2 partial fine-tuning and evaluation
6. EfficientNet-B0 training and evaluation
7. Grad-CAM and DINOv2 attention visualization
8. Results export and interactive inference application


### Environment and hardware check

Import the core libraries and check the Python, PyTorch, Torchvision, and GPU versions. In Colab, select **Runtime → Change runtime type → T4 GPU** before training if GPU access is available.

In [ ]:

import sys
import torch
import torchvision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable one in Runtime settings.")


### Mount Google Drive

Mount Drive so Colab can read the HAM10000 dataset and save model checkpoints and result files persistently. Approve the Google authorization prompt.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Mounted Google Drive.")
except Exception as e:
    print("Not running in Google Colab or Drive mount skipped:", e)


### Inspect Drive folders

List folders in My Drive to help confirm the dataset folder name and troubleshoot incorrect paths before loading the metadata.

In [ ]:

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

print("Folders in My Drive:\n")

for path in DRIVE_ROOT.iterdir():
    if path.is_dir():
        print(path.name)


### Load HAM10000 metadata

Locate and read the HAM10000 metadata CSV. The metadata contains image IDs and diagnostic labels; `DATA_DIR` must point to the folder that contains the CSV and image folders.

In [ ]:

from pathlib import Path
import pandas as pd

from pathlib import Path
import os

# Priority check: Local project directory or Google Drive
POSSIBLE_DATA_DIRS = [
    Path(r"c:/Users/abhin/OneDrive/Desktop/CSE-471/Skin_lesion_datset"),
    Path(r"c:/Users/abhin/OneDrive/Desktop/CSE-471"),
    Path("/content/drive/MyDrive/Skin_lesion_datset"),
    Path("/content/drive/MyDrive/skin_lesion_data"),
]
DATA_DIR = next((d for d in POSSIBLE_DATA_DIRS if d.exists()), Path("/content/drive/MyDrive/Skin_lesion_datset"))
print("Selected DATA_DIR:", DATA_DIR)


# Find the metadata CSV
metadata_files = list(
    DATA_DIR.rglob("HAM10000_metadata.csv")
)

if not metadata_files:
    metadata_files = list(
        DATA_DIR.rglob("HAM10000_metadata")
    )

if not metadata_files:
    raise FileNotFoundError(
        "Metadata file not found. Check DATA_DIR."
    )

metadata = pd.read_csv(metadata_files[0])

print("Metadata file:", metadata_files[0])
print("Dataset shape:", metadata.shape)
print("\nColumns:", metadata.columns.tolist())
print("\nClass distribution:")
print(metadata["dx"].value_counts())


### Count original images and metadata records

Count the original JPG files and metadata records, and print a few filenames. This is a basic integrity check before processing the dataset.

In [ ]:

image_files = list(DATA_DIR.rglob("*.jpg"))

print("Number of original JPG images:", len(image_files))
print("Number of metadata records:", len(metadata))

if image_files:
    print("\nExample images:")
    for image_path in image_files[:5]:
        print(image_path.name)


### Audit image-to-metadata matching

Compare image IDs in the metadata with JPG filenames. This identifies missing images, extra files, and duplicate image IDs.

In [ ]:

from pathlib import Path
import pandas as pd
from collections import defaultdict

# Uses DATA_DIR and metadata from your previous cells
metadata["image_id"] = metadata["image_id"].astype(str)

image_paths = list(DATA_DIR.rglob("*.jpg"))

# Build a mapping from image ID to its file path
images_by_id = defaultdict(list)

for path in image_paths:
    images_by_id[path.stem].append(path)

metadata_ids = set(metadata["image_id"])
available_ids = set(images_by_id)

matched_ids = metadata_ids & available_ids
missing_ids = metadata_ids - available_ids
extra_ids = available_ids - metadata_ids
duplicate_ids = {
    image_id: paths
    for image_id, paths in images_by_id.items()
    if len(paths) > 1
}

print("Metadata records:", len(metadata))
print("JPG files:", len(image_paths))
print("Metadata IDs with images:", len(matched_ids))
print("Metadata IDs without images:", len(missing_ids))
print("Image IDs absent from metadata:", len(extra_ids))
print("IDs with duplicate files:", len(duplicate_ids))

print("\nUnmatched image examples:", list(extra_ids)[:10])


### Explore diagnostic class distribution

Count images in each diagnostic category and plot the class distribution. The imbalance is important when choosing metrics and training strategies.

In [ ]:

import matplotlib.pyplot as plt

class_counts = metadata["dx"].value_counts()

print("Images per class:")
print(class_counts)

plt.figure(figsize=(9, 5))
class_counts.plot(kind="bar")
plt.title("HAM10000: Diagnostic Class Distribution")
plt.xlabel("Diagnosis label")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Visualize example images

Display example dermoscopic images for each diagnostic category to become familiar with the seven HAM10000 labels. These examples are for dataset exploration, not diagnosis.

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

class_names = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions"
}

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()

for ax, label in zip(axes, class_names):
    rows = metadata[metadata["dx"] == label]

    selected_path = None
    selected_id = None

    for image_id in rows["image_id"]:
        if image_id in images_by_id:
            selected_id = image_id
            selected_path = images_by_id[image_id][0]
            break

    if selected_path is not None:
        with Image.open(selected_path) as img:
            ax.imshow(img.convert("RGB"))

        ax.set_title(
            f"{label}: {class_names[label]}\n{selected_id}",
            fontsize=9
        )
    else:
        ax.set_title(f"{label}: image not found")

    ax.axis("off")

axes[-1].axis("off")
plt.tight_layout()
plt.show()


### Build the cleaned metadata table

Match each metadata record to its image path, remove records with missing required fields, and save a clean metadata CSV. The `df` table becomes the basis for splitting the data.

In [ ]:

import pandas as pd
from pathlib import Path

# Match each metadata record to its image
image_map = {}

for path in DATA_DIR.rglob("*.jpg"):
    image_map.setdefault(path.stem, path)

df = metadata.copy()
df["image_path"] = df["image_id"].map(
    lambda image_id: str(image_map[image_id])
    if image_id in image_map else None
)

# Keep only records with an available image and required labels
df = df.dropna(
    subset=["image_path", "dx", "lesion_id"]
).copy()

df["dx"] = df["dx"].astype(str)
df["lesion_id"] = df["lesion_id"].astype(str)

print("Usable image records:", len(df))
print("Unique lesions:", df["lesion_id"].nunique())
print("Diagnostic classes:", df["dx"].nunique())
print("\nClass counts:")
print(df["dx"].value_counts())

df.to_csv(
    "/content/skin_lesion_clean_metadata.csv",
    index=False
)


### Create lesion-aware data splits

Create training, validation, and test partitions using `StratifiedGroupKFold`, grouping by lesion ID to prevent images of the same lesion appearing in multiple partitions. The test set should remain untouched during model selection.

In [ ]:

from sklearn.model_selection import StratifiedGroupKFold

# First, reserve approximately 20% for testing
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_val_idx, test_idx = next(
    outer_splitter.split(
        df,
        y=df["dx"],
        groups=df["lesion_id"]
    )
)

train_val_df = df.iloc[train_val_idx].copy()
test_df = df.iloc[test_idx].copy()

# Split the remaining 80% into approximately 80:20
# for training and validation.
inner_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    inner_splitter.split(
        train_val_df,
        y=train_val_df["dx"],
        groups=train_val_df["lesion_id"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

for name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(
        f"{name}: {len(split_df)} images, "
        f"{split_df['lesion_id'].nunique()} unique lesions"
    )
    print(split_df["dx"].value_counts().sort_index())
    print()

# Verify that no lesion appears in multiple splits
train_groups = set(train_df["lesion_id"])
val_groups = set(val_df["lesion_id"])
test_groups = set(test_df["lesion_id"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("Verified: no lesion ID overlaps between splits.")


### Set device and class mapping

Choose CUDA when available, otherwise CPU. Create a stable mapping from the seven diagnostic labels to integer class indices for training and evaluation.

In [ ]:

import torch
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Training may be slower.")

class_names = sorted(df["dx"].unique())
class_to_idx = {
    name: idx for idx, name in enumerate(class_names)
}

print("Classes:", class_names)
print("Class mapping:", class_to_idx)


### Preprocessing, augmentation, and data loaders

Define image transforms, the custom PyTorch dataset, and data loaders. Random augmentations are applied only to training images; validation and test images use deterministic preprocessing.

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# ImageNet normalization for pretrained ResNet-50
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        with Image.open(row["image_path"]) as img:
            image = img.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[row["dx"]]
        return image, label

train_dataset = SkinLesionDataset(
    train_df, train_transform
)
val_dataset = SkinLesionDataset(
    val_df, eval_transform
)
test_dataset = SkinLesionDataset(
    test_df, eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label examples:", labels[:10].tolist())
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


### Configure the ResNet-50 baseline

Load ImageNet-pretrained ResNet-50, replace its classifier with a seven-class output layer, and configure class-weighted loss, AdamW, and a validation-based learning-rate scheduler.

In [ ]:

import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Download pretrained weights if they are not cached
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Replace the original 1000-class classifier
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))

model = model.to(device)

# Compensate moderately for class imbalance
counts = train_df["dx"].value_counts()
class_weights = torch.tensor(
    [
        (len(train_df) / (len(class_names) * counts[c])) ** 0.5
        for c in class_names
    ],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW([
    {
        "params": [
            p for name, p in model.named_parameters()
            if not name.startswith("fc.")
        ],
        "lr": 1e-4
    },
    {
        "params": model.fc.parameters(),
        "lr": 1e-3
    }
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

print("Model:", type(model).__name__)
print("Output classes:", model.fc.out_features)
print("Device:", device)
print("Class weights:", class_weights.cpu().numpy())


### Copy images to Colab local storage

Copy the two HAM10000 image folders from Drive to Colab's local disk to reduce repeated Drive I/O during training. This needs enough Colab disk space and may take several minutes.

In [ ]:

from pathlib import Path
import shutil

LOCAL_DIR = Path("/content/skin_lesion_data")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for folder_name in [
    "HAM10000_images_part_1",
    "HAM10000_images_part_2"
]:
    matches = list(DATA_DIR.rglob(folder_name))

    if not matches:
        raise FileNotFoundError(f"Cannot find {folder_name}")

    source = matches[0]
    destination = LOCAL_DIR / folder_name

    if not destination.exists():
        print("Copying", folder_name, "...")
        shutil.copytree(source, destination)
    else:
        print("Already copied:", folder_name)

print("Local image copy complete.")


### Reconnect metadata to local images

Build a local image-path lookup and update the train, validation, and test dataframes. Recreate datasets and loaders so training reads images from Colab's local storage.

In [ ]:

# Map image IDs to local files
LOCAL_IMAGES = {
    p.stem: str(p)
    for p in LOCAL_DIR.rglob("*.jpg")
}

print("Local JPG files:", len(LOCAL_IMAGES))

for split_df in [train_df, val_df, test_df]:
    split_df["image_path"] = split_df["image_id"].map(LOCAL_IMAGES)
    assert split_df["image_path"].notna().all()

# Rebuild datasets using the existing transforms
train_dataset = SkinLesionDataset(train_df, train_transform)
val_dataset = SkinLesionDataset(val_df, eval_transform)
test_dataset = SkinLesionDataset(test_df, eval_transform)

# Faster data loading
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


### Run the initial ResNet-50 training epoch

Run one training epoch using mixed precision on CUDA. This cell is the initial training pass; later cells continue training and select a checkpoint using validation macro-F1.

In [ ]:

from tqdm.auto import tqdm
import torch

scaler = torch.amp.GradScaler("cuda")

model.train()
running_loss = 0.0
correct = 0
total = 0

progress = tqdm(train_loader, desc="Optimized training")

for x, y in progress:
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        logits = model(x)
        loss = criterion(logits, y)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    running_loss += loss.item() * y.size(0)
    correct += (logits.argmax(1) == y).sum().item()
    total += y.size(0)

    progress.set_postfix(
        loss=f"{loss.item():.3f}",
        accuracy=f"{correct / total:.3f}"
    )

print("Epoch training loss:", round(running_loss / total, 4))
print("Epoch training accuracy:", round(correct / total, 4))


### Configure optimized data loaders

Rebuild the data loaders with pinned memory and two worker processes. If Colab reports worker warnings or hangs, reduce `NUM_WORKERS` to 0 or 2 and rerun the dataset/loader cells.

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

### Continue ResNet-50 training and save the best checkpoint

Train additional epochs, evaluate on the validation set after each epoch, and save the checkpoint with the best validation macro-F1. The saved path is reused for test evaluation and Grad-CAM.

In [ ]:

import os
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, balanced_accuracy_score

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/resnet50_best.pth"
)

os.makedirs(os.path.dirname(BEST_MODEL_PATH), exist_ok=True)

# Continue from the weights produced by your completed first epoch.
# Keep the existing optimizer and scaler.
NUM_MORE_EPOCHS = 7

best_val_f1 = -1.0
history = []

for epoch in range(NUM_MORE_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in tqdm(
        train_loader,
        desc=f"Epoch {epoch + 2}/8"
    ):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss_sum = 0.0
    val_true, val_pred = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(x)
                loss = criterion(logits, y)

            val_loss_sum += loss.item() * y.size(0)
            val_true.extend(y.cpu().tolist())
            val_pred.extend(logits.argmax(1).cpu().tolist())

    val_loss = val_loss_sum / len(val_dataset)
    val_f1 = f1_score(
        val_true, val_pred,
        average="macro",
        zero_division=0
    )
    val_bal_acc = balanced_accuracy_score(
        val_true, val_pred
    )

    scheduler.step(val_f1)

    history.append({
        "epoch": epoch + 2,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_balanced_accuracy": val_bal_acc
    })

    print(
        f"Epoch {epoch + 2}/8 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val macro-F1: {val_f1:.4f} | "
        f"Val balanced acc: {val_bal_acc:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "epoch": epoch + 2,
            "val_macro_f1": val_f1
        }, BEST_MODEL_PATH)
        print("Saved best model checkpoint.")

print("Additional training complete.")
print("Best validation macro-F1:", round(best_val_f1, 4))


### Reload the selected ResNet-50 checkpoint

Recreate the ResNet-50 architecture and load the saved best checkpoint. This avoids accidentally evaluating the last in-memory epoch instead of the selected checkpoint.

In [ ]:

import torch
import torch.nn as nn
from torchvision import models

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/resnet50_best.pth"
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model = models.resnet50(weights=None)
model.fc = nn.Linear(
    model.fc.in_features,
    len(checkpoint["class_names"])
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Loaded best checkpoint")
print("Saved epoch:", checkpoint["epoch"])
print("Saved validation macro-F1:", checkpoint["val_macro_f1"])


### Evaluate ResNet-50 on the held-out test set

Compute test accuracy, balanced accuracy, macro-F1, class-wise metrics, and macro one-vs-rest AUROC. Use these results for the final model comparison; do not tune model choices against the test set.

In [ ]:

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.metrics import roc_auc_score

model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images)

        probabilities = torch.softmax(logits.float(), dim=1)

        all_labels.extend(labels.numpy().tolist())
        all_preds.extend(
            probabilities.argmax(dim=1).cpu().tolist()
        )
        all_probs.extend(probabilities.cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

class_names = checkpoint["class_names"]

print("Test accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Test balanced accuracy:", round(
    balanced_accuracy_score(y_true, y_pred), 4
))
print("Test macro-F1:", round(
    f1_score(y_true, y_pred, average="macro", zero_division=0), 4
))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

try:
    test_auc = roc_auc_score(
        y_true,
        y_prob,
        multi_class="ovr",
        average="macro",
        labels=list(range(len(class_names)))
    )
    print("Test macro one-vs-rest AUROC:", round(test_auc, 4))
except ValueError as e:
    print("AUROC could not be calculated:", e)


### Plot the ResNet-50 confusion matrix

Visualize true versus predicted labels. Off-diagonal cells show which diagnostic categories the model confuses.

In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names)))
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 8))
display.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=45,
    colorbar=False
)
ax.set_title("ResNet-50 — Test Confusion Matrix")
plt.tight_layout()
plt.show()


# Part 3 — DINOv2 foundation model

DINOv2 is a vision transformer pretrained on a large image collection. We first test frozen feature representations with a logistic-regression classifier, then partially fine-tune the model on HAM10000.

### Load pretrained DINOv2

Load `facebook/dinov2-base` and freeze its parameters for the feature-extraction experiment. The model produces transformer representations for each input image.

In [ ]:

!pip -q install transformers

import torch
from transformers import AutoImageProcessor, AutoModel

MODEL_NAME = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
dinov2 = AutoModel.from_pretrained(MODEL_NAME)

dinov2 = dinov2.to(device)
dinov2.eval()

# Freeze the foundation model initially
for param in dinov2.parameters():
    param.requires_grad = False

print("Foundation model:", MODEL_NAME)
print("Device:", device)
print("Feature dimension:", dinov2.config.hidden_size)


### Create DINOv2 feature-extraction loaders

Create deterministic data loaders for train, validation, and test features. No random augmentation is used so that extracted features are consistent.

In [ ]:

from torch.utils.data import DataLoader

feature_train_dataset = SkinLesionDataset(
    train_df, eval_transform
)
feature_val_dataset = SkinLesionDataset(
    val_df, eval_transform
)
feature_test_dataset = SkinLesionDataset(
    test_df, eval_transform
)

feature_train_loader = DataLoader(
    feature_train_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

feature_val_loader = DataLoader(
    feature_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

feature_test_loader = DataLoader(
    feature_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(feature_train_loader))
print("Validation batches:", len(feature_val_loader))
print("Test batches:", len(feature_test_loader))


### Extract frozen DINOv2 features

Run each image through the frozen DINOv2 backbone and collect its CLS-token representation with the corresponding labels. This avoids repeatedly running the transformer while tuning a small classifier.

In [ ]:

import numpy as np
from tqdm.auto import tqdm

@torch.no_grad()
def extract_features(loader, model, device):
    model.eval()
    features = []
    labels = []

    for images, targets in tqdm(loader, desc="Extracting features"):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(pixel_values=images)
            batch_features = outputs.last_hidden_state[:, 0, :]

        features.append(batch_features.float().cpu())
        labels.append(targets.cpu())

    return torch.cat(features), torch.cat(labels)

X_train, y_train = extract_features(
    feature_train_loader, dinov2, device
)

X_val, y_val = extract_features(
    feature_val_loader, dinov2, device
)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)


### Tune a logistic-regression classifier

Train logistic-regression pipelines with several regularization strengths and select the value with the best validation macro-F1. Only validation data is used for this choice.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.pipeline import make_pipeline

# Convert feature tensors to NumPy
Xtr = X_train.numpy()
Xv = X_val.numpy()
ytr = y_train.numpy()
yv = y_val.numpy()

results = []
best_model = None
best_f1 = -1.0
best_C = None

for C in [0.01, 0.1, 1.0, 10.0]:
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )

    clf.fit(Xtr, ytr)
    val_pred = clf.predict(Xv)

    macro_f1 = f1_score(
        yv, val_pred,
        average="macro",
        zero_division=0
    )
    balanced_acc = balanced_accuracy_score(yv, val_pred)

    results.append({
        "C": C,
        "validation_macro_f1": macro_f1,
        "validation_balanced_accuracy": balanced_acc
    })

    print(
        f"C={C:<5} | "
        f"Macro-F1={macro_f1:.4f} | "
        f"Balanced accuracy={balanced_acc:.4f}"
    )

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_model = clf
        best_C = C

results_df = pd.DataFrame(results)

print("\nSelected C:", best_C)




### Display frozen-feature tuning results

Display the table produced by the preceding tuning cell. If this cell reports that `display` is not callable, use the `show_table` alias already imported here.

In [ ]:

from IPython.display import display as show_table

show_table(results_df)

### Extract DINOv2 test features

Extract frozen DINOv2 features for the held-out test set after choosing the classifier hyperparameter on validation data.

In [ ]:

X_test, y_test = extract_features(
    feature_test_loader, dinov2, device
)

print("Test features:", X_test.shape)
print("Test labels:", y_test.shape)


### Evaluate frozen DINOv2

Evaluate the selected logistic-regression classifier on the test features and report accuracy, balanced accuracy, macro-F1, class-wise metrics, and macro AUROC.

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score
)

Xt = X_test.numpy()
yt = y_test.numpy()

test_pred = best_model.predict(Xt)
test_probs = best_model.predict_proba(Xt)

print("Test accuracy:", round(accuracy_score(yt, test_pred), 4))
print("Test balanced accuracy:",
      round(balanced_accuracy_score(yt, test_pred), 4))
print("Test macro-F1:",
      round(f1_score(yt, test_pred, average="macro",
                     zero_division=0), 4))

print("\nClassification report:")
print(classification_report(
    yt, test_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

print("Macro AUROC:", round(
    roc_auc_score(
        yt, test_probs,
        labels=list(range(len(class_names))),
        multi_class="ovr",
        average="macro"
    ), 4
))


### Configure DINOv2 partial fine-tuning

Freeze the DINOv2 backbone, unfreeze the last two transformer blocks and final normalization layer, and attach a seven-class classification head. Use a smaller learning rate for pretrained weights than for the new head.

In [ ]:

import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, balanced_accuracy_score

# Freeze the entire backbone first
for param in dinov2.parameters():
    param.requires_grad = False

# Unfreeze the final two transformer blocks
for layer in dinov2.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Also train the final normalization layer
for param in dinov2.layernorm.parameters():
    param.requires_grad = True

class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, images):
        outputs = self.backbone(pixel_values=images)
        features = outputs.last_hidden_state[:, 0, :]
        return self.head(features)

fine_tune_model = DINOv2Classifier(
    dinov2, len(class_names)
).to(device)

# Use smaller learning rates for pretrained layers
optimizer_ft = AdamW([
    {
        "params": [
            p for p in dinov2.parameters()
            if p.requires_grad
        ],
        "lr": 1e-5
    },
    {
        "params": fine_tune_model.head.parameters(),
        "lr": 5e-4
    }
], weight_decay=0.01)

# Use the existing class weights from the ResNet experiment
criterion_ft = nn.CrossEntropyLoss(weight=class_weights)

scaler_ft = torch.amp.GradScaler("cuda")

print("Trainable backbone parameters:",
      sum(p.numel() for p in dinov2.parameters()
          if p.requires_grad))
print("Classifier output classes:", len(class_names))


### Fine-tune DINOv2 and save the best checkpoint

Train the classifier for three epochs and choose the checkpoint with the best validation macro-F1. The validation metrics help monitor whether fine-tuning improves the frozen-feature baseline.

In [ ]:

import os

FT_MODEL_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/dinov2_finetuned_best.pth"
)
os.makedirs(os.path.dirname(FT_MODEL_PATH), exist_ok=True)

best_ft_f1 = -1.0
ft_history = []

for epoch in range(3):
    fine_tune_model.train()

    # Keep the frozen backbone in evaluation mode
    fine_tune_model.backbone.eval()
    for layer in fine_tune_model.backbone.encoder.layer[-2:]:
        layer.train()
    fine_tune_model.backbone.layernorm.train()
    fine_tune_model.head.train()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for images, labels in tqdm(
        train_loader, desc=f"Fine-tuning epoch {epoch+1}/3"
    ):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_ft.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda", dtype=torch.float16
        ):
            logits = fine_tune_model(images)
            loss = criterion_ft(logits, labels)

        scaler_ft.scale(loss).backward()
        scaler_ft.step(optimizer_ft)
        scaler_ft.update()

        total_loss += loss.item() * labels.size(0)
        total_correct += (
            logits.argmax(1) == labels
        ).sum().item()
        total += labels.size(0)

    train_loss = total_loss / total
    train_acc = total_correct / total

    # Validation
    fine_tune_model.eval()
    val_true, val_pred = [], []
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, labels in tqdm(
            val_loader, desc="Validation"
        ):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda", dtype=torch.float16
            ):
                logits = fine_tune_model(images)
                loss = criterion_ft(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_true.extend(labels.cpu().tolist())
            val_pred.extend(logits.argmax(1).cpu().tolist())

    val_loss = val_loss_sum / len(val_dataset)
    val_f1 = f1_score(
        val_true, val_pred,
        average="macro", zero_division=0
    )
    val_bal_acc = balanced_accuracy_score(
        val_true, val_pred
    )

    ft_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_balanced_accuracy": val_bal_acc
    })

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val macro-F1: {val_f1:.4f} | "
        f"Val balanced accuracy: {val_bal_acc:.4f}"
    )

    if val_f1 > best_ft_f1:
        best_ft_f1 = val_f1
        torch.save({
            "model_state_dict": fine_tune_model.state_dict(),
            "class_names": class_names,
            "epoch": epoch + 1,
            "val_macro_f1": val_f1
        }, FT_MODEL_PATH)
        print("Saved best fine-tuned model.")

print("Best fine-tuning validation macro-F1:",
      round(best_ft_f1, 4))


### Evaluate fine-tuned DINOv2

Reload the saved fine-tuned checkpoint and evaluate it on the held-out test set. The resulting metrics are compared with ResNet-50, EfficientNet-B0, and frozen DINOv2.

In [ ]:

import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

FT_MODEL_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/dinov2_finetuned_best.pth"
)

checkpoint = torch.load(
    FT_MODEL_PATH,
    map_location=device,
    weights_only=False
)

# Recreate the architecture
dinov2_ft = AutoModel.from_pretrained("facebook/dinov2-base")

for p in dinov2_ft.parameters():
    p.requires_grad = False

for layer in dinov2_ft.encoder.layer[-2:]:
    for p in layer.parameters():
        p.requires_grad = True

for p in dinov2_ft.layernorm.parameters():
    p.requires_grad = True

fine_tune_model = DINOv2Classifier(
    dinov2_ft, len(class_names)
).to(device)

fine_tune_model.load_state_dict(
    checkpoint["model_state_dict"]
)
fine_tune_model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])
print("Validation macro-F1:", checkpoint["val_macro_f1"])

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Fine-tuned DINOv2 test"):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda", dtype=torch.float16
        ):
            logits = fine_tune_model(images)

        probs = torch.softmax(logits.float(), dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(probs.argmax(1).cpu().tolist())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("\nTest accuracy:",
      round(accuracy_score(y_true, y_pred), 4))
print("Test balanced accuracy:",
      round(balanced_accuracy_score(y_true, y_pred), 4))
print("Test macro-F1:",
      round(f1_score(
          y_true, y_pred,
          average="macro", zero_division=0
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

print("Macro AUROC:", round(
    roc_auc_score(
        y_true, y_prob,
        labels=list(range(len(class_names))),
        multi_class="ovr",
        average="macro"
    ), 4
))

cm = confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(class_names)))
)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title("Fine-tuned DINOv2 — Test Confusion Matrix")
plt.tight_layout()
plt.show()


# Part 4 — EfficientNet-B0 CNN baseline

EfficientNet-B0 is a second CNN architecture used to compare performance and computational efficiency with ResNet-50 and DINOv2.

### Initialize EfficientNet-B0

Load ImageNet-pretrained EfficientNet-B0 and replace its classifier with a seven-class output layer.

In [ ]:

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.EfficientNet_B0_Weights.DEFAULT
efficientnet = models.efficientnet_b0(weights=weights)

# Replace the original 1000-class output layer
in_features = efficientnet.classifier[1].in_features
efficientnet.classifier[1] = nn.Linear(
    in_features, len(class_names)
)

efficientnet = efficientnet.to(device)

print("Model: EfficientNet-B0")
print("Output classes:", efficientnet.classifier[1].out_features)
print("Device:", device)


### Configure EfficientNet-B0 optimization

Use class-weighted cross-entropy and AdamW with a smaller learning rate for the pretrained feature extractor and a larger learning rate for the new classifier.

In [ ]:

import torch

criterion_eff = nn.CrossEntropyLoss(weight=class_weights)

optimizer_eff = torch.optim.AdamW([
    {
        "params": [
            p for name, p in efficientnet.named_parameters()
            if not name.startswith("classifier.")
        ],
        "lr": 1e-4
    },
    {
        "params": efficientnet.classifier.parameters(),
        "lr": 1e-3
    }
], weight_decay=1e-4)

scaler_eff = torch.amp.GradScaler("cuda")

print("EfficientNet-B0 training configuration ready.")


### Train EfficientNet-B0 and save the best checkpoint

Train for five epochs and save the checkpoint with the best validation macro-F1. This selected checkpoint is used for the final test evaluation.

In [ ]:

import os
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, balanced_accuracy_score

EFF_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/efficientnet_b0_best.pth"
)
os.makedirs(os.path.dirname(EFF_PATH), exist_ok=True)

best_eff_f1 = -1.0
eff_history = []

for epoch in range(5):
    efficientnet.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in tqdm(
        train_loader, desc=f"EfficientNet epoch {epoch+1}/5"
    ):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer_eff.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda", dtype=torch.float16
        ):
            logits = efficientnet(x)
            loss = criterion_eff(logits, y)

        scaler_eff.scale(loss).backward()
        scaler_eff.step(optimizer_eff)
        scaler_eff.update()

        running_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    efficientnet.eval()
    val_loss_sum = 0.0
    val_true, val_pred = [], []

    with torch.no_grad():
        for x, y in tqdm(val_loader, desc="Validation"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda", dtype=torch.float16
            ):
                logits = efficientnet(x)
                loss = criterion_eff(logits, y)

            val_loss_sum += loss.item() * y.size(0)
            val_true.extend(y.cpu().tolist())
            val_pred.extend(logits.argmax(1).cpu().tolist())

    val_loss = val_loss_sum / len(val_dataset)
    val_f1 = f1_score(
        val_true, val_pred,
        average="macro", zero_division=0
    )
    val_bal_acc = balanced_accuracy_score(val_true, val_pred)

    eff_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_balanced_accuracy": val_bal_acc
    })

    print(
        f"Epoch {epoch+1}/5 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val macro-F1: {val_f1:.4f} | "
        f"Val balanced acc: {val_bal_acc:.4f}"
    )

    if val_f1 > best_eff_f1:
        best_eff_f1 = val_f1
        torch.save({
            "model_state_dict": efficientnet.state_dict(),
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "epoch": epoch + 1,
            "val_macro_f1": val_f1
        }, EFF_PATH)
        print("Saved best EfficientNet-B0 checkpoint.")

print("Best validation macro-F1:", round(best_eff_f1, 4))


### Evaluate EfficientNet-B0

Load the best EfficientNet-B0 checkpoint and compute test accuracy, balanced accuracy, macro-F1, class-wise precision/recall/F1, macro AUROC, and a confusion matrix.

In [ ]:

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from tqdm.auto import tqdm

EFF_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/efficientnet_b0_best.pth"
)

checkpoint = torch.load(
    EFF_PATH,
    map_location=device,
    weights_only=False
)

# Recreate the exact architecture
eff_weights = models.EfficientNet_B0_Weights.DEFAULT
efficientnet = models.efficientnet_b0(weights=None)

in_features = efficientnet.classifier[1].in_features
efficientnet.classifier[1] = nn.Linear(
    in_features, len(checkpoint["class_names"])
)

efficientnet.load_state_dict(checkpoint["model_state_dict"])
efficientnet = efficientnet.to(device)
efficientnet.eval()

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Best validation macro-F1:", checkpoint["val_macro_f1"])

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="EfficientNet test"):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = efficientnet(images)

        probs = torch.softmax(logits.float(), dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(probs.argmax(dim=1).cpu().tolist())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)
class_names = checkpoint["class_names"]

print("\nEfficientNet-B0 test accuracy:",
      round(accuracy_score(y_true, y_pred), 4))
print("Test balanced accuracy:",
      round(balanced_accuracy_score(y_true, y_pred), 4))
print("Test macro-F1:",
      round(f1_score(
          y_true, y_pred,
          average="macro", zero_division=0
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

print("Macro AUROC:", round(
    roc_auc_score(
        y_true, y_prob,
        labels=list(range(len(class_names))),
        multi_class="ovr",
        average="macro"
    ), 4
))

cm = confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(class_names)))
)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(ax=ax, xticks_rotation=45, colorbar=False)

ax.set_title("EfficientNet-B0 — Test Confusion Matrix")
plt.tight_layout()
plt.show()


# Part 5 — Explainability

The next cells generate Grad-CAM overlays for the two CNNs and attention visualizations for DINOv2. Review several correct and incorrect examples; do not interpret an attention map as a clinically reliable explanation.

### Load CNN checkpoints for explainability

Load the saved ResNet-50 and EfficientNet-B0 checkpoints so that Grad-CAM uses the selected models rather than an arbitrary training epoch.

In [ ]:

import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from pathlib import Path
POSSIBLE_MODEL_DIRS = [
    Path(r"c:/Users/abhin/OneDrive/Desktop/CSE-471/skin_lesion_app/models"),
    Path("/content/drive/MyDrive/skin_lesion_project/models"),
]
MODEL_DIR = next((d for d in POSSIBLE_MODEL_DIRS if d.exists()), Path(r"c:/Users/abhin/OneDrive/Desktop/CSE-471/skin_lesion_app/models"))
print("Selected MODEL_DIR:", MODEL_DIR)


RESNET_PATH = MODEL_DIR / "resnet50_best.pth"
EFF_PATH = MODEL_DIR / "efficientnet_b0_best.pth"

for path in [RESNET_PATH, EFF_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {path}. "
            "Check that Google Drive is mounted and the file exists."
        )

# Load ResNet-50
resnet_ckpt = torch.load(
    RESNET_PATH, map_location=device, weights_only=False
)

resnet_model = models.resnet50(weights=None)
resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    len(resnet_ckpt["class_names"])
)
resnet_model.load_state_dict(resnet_ckpt["model_state_dict"])
resnet_model = resnet_model.to(device).eval()

# Load EfficientNet-B0
eff_ckpt = torch.load(
    EFF_PATH, map_location=device, weights_only=False
)

eff_model = models.efficientnet_b0(weights=None)
eff_model.classifier[1] = nn.Linear(
    eff_model.classifier[1].in_features,
    len(eff_ckpt["class_names"])
)
eff_model.load_state_dict(eff_ckpt["model_state_dict"])
eff_model = eff_model.to(device).eval()

class_names = resnet_ckpt["class_names"]

assert class_names == eff_ckpt["class_names"]

print("ResNet-50 loaded from epoch:", resnet_ckpt["epoch"])
print("EfficientNet-B0 loaded from epoch:", eff_ckpt["epoch"])
print("Classes:", class_names)


### Collect CNN test predictions

Generate test predictions for both CNNs and identify correctly and incorrectly classified melanoma cases. These examples are used for qualitative explainability analysis.

In [ ]:

import numpy as np
from torch.utils.data import DataLoader

test_eval_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

mel_idx = class_names.index("mel")

# Get predictions from both models
resnet_model.eval()
eff_model.eval()

resnet_preds = []
eff_preds = []
true_labels = []

with torch.no_grad():
    for images, labels in test_eval_loader:
        images = images.to(device)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            r_logits = resnet_model(images)
            e_logits = eff_model(images)

        resnet_preds.extend(r_logits.argmax(1).cpu().tolist())
        eff_preds.extend(e_logits.argmax(1).cpu().tolist())
        true_labels.extend(labels.tolist())

resnet_preds = np.array(resnet_preds)
eff_preds = np.array(eff_preds)
true_labels = np.array(true_labels)

mel_indices = np.where(true_labels == mel)[0]

resnet_correct_mel = [
    i for i in mel_indices if resnet_preds[i] == mel
]
resnet_wrong_mel = [
    i for i in mel_indices if resnet_preds[i] != mel
]
eff_correct_mel = [
    i for i in mel_indices if eff_preds[i] == mel
]
eff_wrong_mel = [
    i for i in mel_indices if eff_preds[i] != mel
]

print("Melanoma test images:", len(mel_indices))
print("ResNet-50 correct:", len(resnet_correct_mel))
print("ResNet-50 incorrect:", len(resnet_wrong_mel))
print("EfficientNet-B0 correct:", len(eff_correct_mel))
print("EfficientNet-B0 incorrect:", len(eff_wrong_mel))


### Prepare melanoma case indices

Create lists of correctly and incorrectly classified melanoma examples for each CNN. The class index is derived from `class_names`, avoiding hard-coded numeric labels.

In [ ]:

import numpy as np

# Find the class index for melanoma
mel_idx = class_names.index("mel")

# Use the true labels generated in the previous cell
true_labels = np.array(true_labels)
resnet_preds = np.array(resnet_preds)
eff_preds = np.array(eff_preds)

mel_indices = np.where(true_labels == mel_idx)[0]

resnet_correct_mel = [
    i for i in mel_indices if resnet_preds[i] == mel_idx
]
resnet_wrong_mel = [
    i for i in mel_indices if resnet_preds[i] != mel_idx
]

eff_correct_mel = [
    i for i in mel_indices if eff_preds[i] == mel_idx
]
eff_wrong_mel = [
    i for i in mel_indices if eff_preds[i] != mel_idx
]

print("Melanoma class index:", mel_idx)
print("Melanoma test images:", len(mel_indices))
print("ResNet-50 correct:", len(resnet_correct_mel))
print("ResNet-50 incorrect:", len(resnet_wrong_mel))
print("EfficientNet-B0 correct:", len(eff_correct_mel))
print("EfficientNet-B0 incorrect:", len(eff_wrong_mel))


### Define Grad-CAM visualization functions

Implement Grad-CAM for the final convolutional layer of each CNN, then create overlays showing the original image, heatmap, and prediction. The heatmap is a model-attribution aid, not proof of clinical reasoning.

In [ ]:

import torch.nn.functional as F
import matplotlib.pyplot as plt

def generate_gradcam(model, target_layer, image_tensor):
    activations = {}
    gradients = {}

    def forward_hook(module, inputs, output):
        activations["value"] = output
        output.register_hook(
            lambda grad: gradients.update(value=grad)
        )

    handle = target_layer.register_forward_hook(forward_hook)

    try:
        model.eval()
        model.zero_grad(set_to_none=True)

        x = image_tensor.unsqueeze(0).to(device)

        # FP32 is used here for a straightforward gradient calculation
        with torch.enable_grad():
            logits = model(x)
            predicted_class = logits.argmax(dim=1).item()
            score = logits[0, predicted_class]
            score.backward()

        feature_map = activations["value"]
        gradient = gradients["value"]

        # Average gradients over spatial dimensions
        weights = gradient.mean(dim=(2, 3), keepdim=True)

        cam = (weights * feature_map).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(
            cam,
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        )

        cam = cam[0, 0]
        cam = cam / (cam.max() + 1e-8)

        probabilities = torch.softmax(logits.detach(), dim=1)[0]

        return (
            cam.detach().cpu().numpy(),
            predicted_class,
            probabilities.cpu().numpy()
        )
    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)


# Undo ImageNet normalization for visualization
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])

def display_gradcam(model, target_layer, dataset, index, model_name):
    image_tensor, true_label = dataset[index]

    cam, predicted_class, probs = generate_gradcam(
        model, target_layer, image_tensor
    )

    image = image_tensor.permute(1, 2, 0).numpy()
    image = np.clip(image * STD + MEAN, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].imshow(image)
    axes[0].set_title(
        f"Original\nTrue: {class_names[true_label]}"
    )

    axes[1].imshow(cam, cmap="jet", vmin=0, vmax=1)
    axes[1].set_title("Grad-CAM heatmap")

    axes[2].imshow(image)
    axes[2].imshow(
        cam, cmap="jet", alpha=0.40, vmin=0, vmax=1
    )
    axes[2].set_title(
        f"{model_name}\nPredicted: {class_names[predicted_class]}"
        f"\nConfidence: {probs[predicted_class]:.1%}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("True class:", class_names[true_label])
    print("Predicted class:", class_names[predicted_class])
    print("Prediction probabilities:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {probs[i]:.4f}")


# Final convolutional layers
resnet_target = resnet_model.layer4[-1]
eff_target = eff_model.features[-1]

print("Grad-CAM functions ready.")


### Visualize ResNet-50 melanoma cases

Display Grad-CAM for a correctly classified melanoma image and a misclassified melanoma image, if both are available.

In [ ]:

if resnet_correct_mel:
    display_gradcam(
        resnet_model,
        resnet_target,
        test_dataset,
        resnet_correct_mel[0],
        "ResNet-50"
    )

if resnet_wrong_mel:
    display_gradcam(
        resnet_model,
        resnet_target,
        test_dataset,
        resnet_wrong_mel[0],
        "ResNet-50"
    )


### Visualize EfficientNet-B0 melanoma cases

Display Grad-CAM for EfficientNet-B0 on correctly and incorrectly classified melanoma examples.

In [ ]:

if eff_correct_mel:
    display_gradcam(
        eff_model,
        eff_target,
        test_dataset,
        eff_correct_mel[0],
        "EfficientNet-B0"
    )

if eff_wrong_mel:
    display_gradcam(
        eff_model,
        eff_target,
        test_dataset,
        eff_wrong_mel[0],
        "EfficientNet-B0"
    )


### Generate a combined CNN Grad-CAM review

Display both CNNs' Grad-CAM examples in sequence to make qualitative comparisons easier.

In [ ]:

if resnet_correct_mel:
    display_gradcam(
        resnet_model, resnet_target, test_dataset,
        resnet_correct_mel[0], "ResNet-50"
    )

if resnet_wrong_mel:
    display_gradcam(
        resnet_model, resnet_target, test_dataset,
        resnet_wrong_mel[0], "ResNet-50"
    )

if eff_correct_mel:
    display_gradcam(
        eff_model, eff_target, test_dataset,
        eff_correct_mel[0], "EfficientNet-B0"
    )

if eff_wrong_mel:
    display_gradcam(
        eff_model, eff_target, test_dataset,
        eff_wrong_mel[0], "EfficientNet-B0"
    )


### Load fine-tuned DINOv2 for attention visualization

Load the fine-tuned DINOv2 checkpoint with eager attention enabled so attention matrices can be retrieved for visualization.

In [ ]:

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "facebook/dinov2-base"
FT_MODEL_PATH = (
    "/content/drive/MyDrive/skin_lesion_project/"
    "models/dinov2_finetuned_best.pth"
)

# Eager attention allows us to retrieve attention matrices
dinov2_attention_backbone = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
).to(device)

class DINOv2AttentionClassifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(
            backbone.config.hidden_size, num_classes
        )

    def forward(self, images, output_attentions=False):
        outputs = self.backbone(
            pixel_values=images,
            output_attentions=output_attentions
        )
        features = outputs.last_hidden_state[:, 0, :]
        logits = self.head(features)
        return logits, outputs

dinov2_explain_model = DINOv2AttentionClassifier(
    dinov2_attention_backbone, len(class_names)
).to(device)

checkpoint = torch.load(
    FT_MODEL_PATH,
    map_location=device,
    weights_only=False
)

dinov2_explain_model.load_state_dict(
    checkpoint["model_state_dict"]
)
dinov2_explain_model.eval()

print("Loaded fine-tuned DINOv2 checkpoint")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Device:", device)


### Extract DINOv2 CLS-to-patch attention

Average the final transformer block's attention across heads, select CLS-token-to-patch attention, and resize the patch grid for display. This is an exploratory attention map, not a complete explanation of the prediction.

In [ ]:

import torch.nn.functional as F

def generate_dinov2_attention(model, image_tensor):
    model.eval()

    image_batch = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits, outputs = model(
            image_batch,
            output_attentions=True
        )

    attentions = outputs.attentions

    if attentions is None or attentions[-1] is None:
        raise RuntimeError(
            "Attention matrices were not returned. "
            "Ensure the backbone was loaded with "
            "attn_implementation='eager'."
        )

    # Shape: [batch, heads, tokens, tokens]
    last_attention = attentions[-1][0]

    # Average attention heads, then select CLS-to-patch attention
    cls_attention = last_attention.mean(dim=0)[0, 1:]

    # DINOv2-base uses 14x14-pixel patches
    patch_size = model.backbone.config.patch_size
    height, width = image_tensor.shape[-2:]

    grid_h = height // patch_size
    grid_w = width // patch_size

    if cls_attention.numel() != grid_h * grid_w:
        raise ValueError(
            f"Unexpected patch count: {cls_attention.numel()}"
        )

    attention_map = cls_attention.reshape(
        1, 1, grid_h, grid_w
    )

    # Upsample the patch-level map for visualization
    attention_map = F.interpolate(
        attention_map,
        size=(height, width),
        mode="bilinear",
        align_corners=False
    )[0, 0]

    attention_map = attention_map.cpu().numpy()
    attention_map = (
        attention_map - attention_map.min()
    ) / (
        attention_map.max() - attention_map.min() + 1e-8
    )

    probabilities = torch.softmax(logits.float(), dim=1)[0]

    return attention_map, probabilities.cpu().numpy()


### Visualize one DINOv2 attention map

Show the original image, attention heatmap, and overlay for a selected test image, together with the true label and model prediction.

In [ ]:

def show_dinov2_attention(dataset, sample_idx):
    image_tensor, true_label = dataset[sample_idx]

    attention_map, probabilities = generate_dinov2_attention(
        dinov2_explain_model,
        image_tensor
    )

    predicted_idx = int(np.argmax(probabilities))

    # Undo ImageNet normalization for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image_tensor.permute(1, 2, 0).numpy()
    image = np.clip(image * std + mean, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(image)
    axes[0].set_title(
        f"Original image\nTrue: {class_names[true_label]}"
    )

    axes[1].imshow(
        attention_map, cmap="jet", vmin=0, vmax=1
    )
    axes[1].set_title("DINOv2 CLS attention")

    axes[2].imshow(image)
    axes[2].imshow(
        attention_map, cmap="jet", alpha=0.4,
        vmin=0, vmax=1
    )
    axes[2].set_title(
        f"Attention overlay\nPredicted: {class_names[predicted_idx]}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("True class:", class_names[true_label])
    print("Predicted class:", class_names[predicted_idx])
    print("Predicted-class probability:",
          f"{probabilities[predicted_idx]:.4f}")


# Display one test example
show_dinov2_attention(test_dataset, 0)


### Find correct and incorrect DINOv2 melanoma cases

Generate fine-tuned DINOv2 test predictions and identify melanoma cases classified correctly and incorrectly. These cases are used in the attention gallery.

In [ ]:

# Reuse the earlier arrays if they still exist.
# Otherwise, generate predictions using the fine-tuned model.
if "true_labels" not in globals() or "eff_preds" not in globals():
    raise RuntimeError(
        "Run the earlier prediction cell first to create "
        "true_labels, resnet_preds, and eff_preds."
    )

# Generate fine-tuned DINOv2 predictions on the test set
dinov2_preds = []

dinov2_explain_model.eval()

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="DINOv2 predictions"):
        images = images.to(device)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits, _ = dinov2_explain_model(images)

        dinov2_preds.extend(logits.argmax(1).cpu().tolist())

dinov2_preds = np.array(dinov2_preds)
true_labels = np.array(true_labels)

mel_idx = class_names.index("mel")
mel_indices = np.where(true_labels == mel_idx)[0]

correct_mel = [
    i for i in mel_indices if dinov2_preds[i] == mel_idx
]
incorrect_mel = [
    i for i in mel_indices if dinov2_preds[i] != mel_idx
]

print("Correct melanoma predictions:", len(correct_mel))
print("Incorrect melanoma predictions:", len(incorrect_mel))

if correct_mel:
    show_dinov2_attention(test_dataset, correct_mel[0])

if incorrect_mel:
    show_dinov2_attention(test_dataset, incorrect_mel[0])


### Build and save the DINOv2 attention gallery

Create a gallery of up to three correctly classified and three misclassified melanoma examples, then save it to the project results folder.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Select melanoma cases from the held-out test set
mel_idx = class_names.index("mel")
mel_indices = np.where(true_labels == mel_idx)[0]

correct_indices = [
    i for i in mel_indices if dinov2_preds[i] == mel_idx
][:3]

incorrect_indices = [
    i for i in mel_indices if dinov2_preds[i] != mel_idx
][:3]

selected_indices = correct_indices + incorrect_indices

if not selected_indices:
    raise ValueError("No melanoma examples found in the test predictions.")

fig, axes = plt.subplots(
    len(selected_indices), 3,
    figsize=(12, 4 * len(selected_indices)),
    squeeze=False
)

for row_idx, sample_idx in enumerate(selected_indices):
    image_tensor, true_label = test_dataset[sample_idx]

    attention_map, probabilities = generate_dinov2_attention(
        dinov2_explain_model, image_tensor
    )

    pred_idx = int(np.argmax(probabilities))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image_tensor.permute(1, 2, 0).numpy()
    image = np.clip(image * std + mean, 0, 1)

    status = "Correct" if pred_idx == true_label else "Misclassified"

    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(
        f"{status}\nTrue: {class_names[true_label]}"
    )

    axes[row_idx, 1].imshow(
        attention_map, cmap="jet", vmin=0, vmax=1
    )
    axes[row_idx, 1].set_title("DINOv2 attention")

    axes[row_idx, 2].imshow(image)
    axes[row_idx, 2].imshow(
        attention_map, cmap="jet", alpha=0.4,
        vmin=0, vmax=1
    )
    axes[row_idx, 2].set_title(
        f"Predicted: {class_names[pred_idx]}\n"
        f"Probability: {probabilities[pred_idx]:.1%}"
    )

    for ax in axes[row_idx]:
        ax.axis("off")

plt.suptitle(
    "DINOv2 Attention Analysis — Melanoma Test Images",
    fontsize=15
)
plt.tight_layout()

output_dir = Path(
    "/content/drive/MyDrive/skin_lesion_project/results"
)
output_dir.mkdir(parents=True, exist_ok=True)

gallery_path = output_dir / "dinov2_melanoma_attention_gallery.png"
plt.savefig(gallery_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved gallery to:", gallery_path)
print("Correct examples:", len(correct_indices))
print("Misclassified examples:", len(incorrect_indices))


# Part 6 — Results and artifacts

Check that checkpoints and visualizations exist in Google Drive. The final chart uses the reported test metrics from the experiments.

### Verify saved outputs and checkpoints

List files in the project results and model directories to verify that visualizations, result tables, and trained checkpoints were saved.

In [ ]:

from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/skin_lesion_project/results"
)
MODEL_DIR = Path(
    "/content/drive/MyDrive/skin_lesion_project/models"
)

print("=== RESULTS ===")
for path in sorted(RESULTS_DIR.glob("*")):
    print(f"{path.name} — {path.stat().st_size / 1024:.1f} KB")

print("\n=== MODEL CHECKPOINTS ===")
for path in sorted(MODEL_DIR.glob("*")):
    print(f"{path.name} — {path.stat().st_size / (1024**2):.2f} MB")


### Plot the model comparison chart

Plot the reported test metrics for ResNet-50, EfficientNet-B0, frozen DINOv2, and fine-tuned DINOv2. These values summarize the experiments already completed; rerun evaluations if checkpoints or data splits change.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

metrics = ["Test Accuracy", "Balanced Accuracy", "Macro F1", "Macro AUROC"]

model_results = {
    "ResNet-50": [0.8242, 0.6553, 0.6519, 0.9558],
    "EfficientNet-B0": [0.8133, 0.6565, 0.6431, 0.9570],
    "DINOv2 Frozen": [0.7393, 0.6246, 0.5877, 0.9181],
    "DINOv2 Fine-tuned": [0.7582, 0.6342, 0.5920, 0.9374]
}

x = np.arange(len(metrics))
width = 0.19

fig, ax = plt.subplots(figsize=(12, 6))

for i, (name, scores) in enumerate(model_results.items()):
    ax.bar(
        x + (i - 1.5) * width,
        scores,
        width,
        label=name
    )

ax.set_title("Skin Lesion Classification: Model Comparison")
ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=10)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

chart_path = RESULTS_DIR / "model_comparison.png"
plt.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()

print("Comparison chart saved to:", chart_path)


# Part 7 — Interactive prediction application

This optional section creates a simple Gradio interface. A user uploads a dermoscopic image, selects one of the saved models, and receives the predicted HAM10000 category plus scores for all seven classes.

**Before running:** confirm that Google Drive is mounted and the three checkpoints exist in `skin_lesion_project/models`. The app is designed for images similar to HAM10000 dermoscopic images. It is not a medical diagnostic tool.

### Install the interface dependency

Install Gradio for the upload interface. If it is already installed, this cell should finish quickly.

In [ ]:
!pip -q install gradio transformers


### Load checkpoints for inference

Recreate each architecture and load the saved weights. The DINOv2 wrapper must match the architecture used during fine-tuning: a DINOv2 backbone with a linear head over the CLS token.

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from pathlib import Path
POSSIBLE_MODEL_DIRS = [
    Path(r"c:/Users/abhin/OneDrive/Desktop/CSE-471/skin_lesion_app/models"),
    Path("/content/drive/MyDrive/skin_lesion_project/models"),
]
MODEL_DIR = next((d for d in POSSIBLE_MODEL_DIRS if d.exists()), Path(r"c:/Users/abhin/OneDrive/Desktop/CSE-471/skin_lesion_app/models"))
print("Selected MODEL_DIR:", MODEL_DIR)


CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_DESCRIPTIONS = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}

def load_checkpoint(path):
    if not path.exists():
        print(f"Warning: Checkpoint not found: {path}. Skipping."); return None
    return torch.load(path, map_location="cpu", weights_only=False)

# ResNet-50
resnet_ckpt = load_checkpoint(MODEL_DIR / "resnet50_best.pth")
resnet_model = models.resnet50(weights=None)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, len(resnet_ckpt["class_names"]))
resnet_model.load_state_dict(resnet_ckpt["model_state_dict"])
resnet_model = resnet_model.to(device).eval()

# EfficientNet-B0
eff_ckpt = load_checkpoint(MODEL_DIR / "efficientnet_b0_best.pth")
efficientnet_model = models.efficientnet_b0(weights=None)
efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    len(eff_ckpt["class_names"])
)
efficientnet_model.load_state_dict(eff_ckpt["model_state_dict"])
efficientnet_model = efficientnet_model.to(device).eval()

# DINOv2 fine-tuned classifier
class DINOv2ClassifierForApp(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.config.hidden_size, num_classes)

    def forward(self, images):
        outputs = self.backbone(pixel_values=images)
        return self.head(outputs.last_hidden_state[:, 0, :])

dino_ckpt = load_checkpoint(MODEL_DIR / "dinov2_finetuned_best.pth")
dino_backbone = AutoModel.from_pretrained("facebook/dinov2-base")
dinov2_model = DINOv2ClassifierForApp(dino_backbone, len(dino_ckpt["class_names"]))
dinov2_model.load_state_dict(dino_ckpt["model_state_dict"])
dinov2_model = dinov2_model.to(device).eval()

# Use the saved label order, and check that all models agree.
class_names = resnet_ckpt["class_names"]
assert class_names == eff_ckpt["class_names"] == dino_ckpt["class_names"], \
    "The saved checkpoints use different class orders."

print("All three checkpoints loaded.")
print("Classes:", class_names)
print("Inference device:", device)


### Define image preprocessing and prediction

The uploaded image is converted to RGB, resized to 224 × 224, and normalized using ImageNet statistics. The returned scores are model softmax scores; they are not calibrated disease probabilities.

In [ ]:
import numpy as np
from PIL import Image
from torchvision import transforms
import gradio as gr

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

@torch.inference_mode()
def predict_skin_lesion(image, model_choice):
    if image is None:
        raise gr.Error("Please upload a dermoscopic image.")

    image = image.convert("RGB")
    image_tensor = inference_transform(image).unsqueeze(0).to(device)

    if model_choice == "ResNet-50":
        logits = resnet_model(image_tensor)
    elif model_choice == "EfficientNet-B0":
        logits = efficientnet_model(image_tensor)
    elif model_choice == "DINOv2 Fine-tuned":
        logits = dinov2_model(image_tensor)
    else:
        raise gr.Error("Select a supported model.")

    scores = torch.softmax(logits.float(), dim=1)[0].cpu().numpy()
    score_dict = {
        f"{class_names[i]} — {CLASS_DESCRIPTIONS[class_names[i]]}": float(scores[i])
        for i in range(len(class_names))
    }

    best_idx = int(np.argmax(scores))
    summary = (
        f"Selected model: {model_choice}\n"
        f"Predicted category: {class_names[best_idx]}\n"
        f"Category description: {CLASS_DESCRIPTIONS[class_names[best_idx]]}\n"
        f"Model score: {scores[best_idx]:.1%}\n\n"
        "Research disclaimer: This output is an experimental classification result, "
        "not a medical diagnosis. The model may be wrong, and the score is not a "
        "calibrated probability of disease."
    )
    return score_dict, summary


### Launch the upload interface

Run this cell after the model-loading and prediction cells. Gradio displays a temporary share link in Colab when `share=True`; treat that link as accessible to anyone who has it and do not upload private patient images.

In [ ]:
with gr.Blocks(title="Skin Lesion Research Classifier") as demo:
    gr.Markdown(
        """
        # Skin Lesion Classification
        **Research prototype | HAM10000 seven-class image classification**

        Upload a dermoscopic image and compare predictions from the trained models.
        """
    )

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload dermoscopic image")
            model_input = gr.Dropdown(
                choices=["ResNet-50", "EfficientNet-B0", "DINOv2 Fine-tuned"],
                value="ResNet-50",
                label="Model"
            )
            predict_button = gr.Button("Analyze image", variant="primary")

        with gr.Column():
            scores_output = gr.Label(num_top_classes=7, label="Class scores")
            summary_output = gr.Textbox(label="Prediction summary", lines=7)

    predict_button.click(
        fn=predict_skin_lesion,
        inputs=[image_input, model_input],
        outputs=[scores_output, summary_output]
    )

    gr.Markdown(
        """
        **Medical disclaimer:** This application is for educational research only.
        It has not been validated for clinical use. Do not use it to diagnose or
        rule out skin cancer; consult a qualified healthcare professional about
        concerning skin lesions.
        """
    )

demo.launch(share=True, debug=True)


# Final notes and limitations

## Reported experimental results

| Model configuration | Test accuracy | Balanced accuracy | Macro-F1 | Macro AUROC |
|---|---:|---:|---:|---:|
| ResNet-50 | 82.42% | 65.53% | 65.19% | 95.58% |
| EfficientNet-B0 | 81.33% | 65.65% | 64.31% | 95.70% |
| DINOv2 frozen features + logistic regression | 73.93% | 62.46% | 58.77% | 91.81% |
| DINOv2 partial fine-tuning | 75.82% | 63.42% | 59.20% | 93.74% |

These are the reported results from the completed experiments and should be verified against the saved outputs before publication. The reported checkpoint-selection procedures were not perfectly uniform across all models, so comparisons should be presented with that limitation.

## Limitations
- HAM10000 is class-imbalanced, and rare classes have few test examples.
- Results on a held-out split from the same dataset do not establish generalization to other hospitals, cameras, skin tones, or ordinary phone photographs.
- DINOv2 attention and Grad-CAM are qualitative attribution tools, not proof of causal reasoning or clinical reliability.
- Softmax scores are not calibrated probabilities of disease.
- This is an educational research prototype, not a clinically validated diagnostic system.

## Suggested future work
- Use consistent validation-based checkpoint selection for every model.
- Report confidence intervals and perform external validation on an independent dataset.
- Investigate class imbalance and calibration without repeatedly tuning on the test set.
- Compare explanations with dermatologist-annotated lesion masks if such annotations are available.
- Add a user interface only for research demonstrations, with clear safety messaging.